# 10 — Tokenization
### Starter Notebook

**Learning objectives**
- Understand why character/word tokenisation falls short
- Implement a simple BPE merge algorithm from scratch
- Use SentencePiece / HuggingFace tokenisers
- Understand special tokens, padding, and attention masks

---

In [ ]:
import sys, os, re
sys.path.insert(0, os.path.abspath('../..'))
from collections import Counter, defaultdict
import torch
import matplotlib.pyplot as plt

print('Imports OK')

## Part A — Why Not Characters or Words?

| Method | Vocab size | Sequence length | OOV? |
|--------|-----------|-----------------|------|
| Character | ~100 | Very long | No |
| Word | 100k–1M | Short | Yes |
| BPE (subword) | 30k–100k | Medium | No |

BPE balances all three dimensions.

In [ ]:
sample = "the transformer is a powerful architecture for natural language processing"

char_tokens = list(sample)
word_tokens = sample.split()

print(f'Character tokenisation: {len(char_tokens)} tokens')
print(f'Word tokenisation     : {len(word_tokens)} tokens')
print()
print(f'Characters: {char_tokens[:15]}...')
print(f'Words     : {word_tokens}')

## Part B — BPE from Scratch

Byte-Pair Encoding iteratively merges the most frequent pair of adjacent symbols.

### Exercise B1 — Implement the core merge step

In [ ]:
def get_vocab(corpus: list[str]) -> dict:
    """
    Build initial vocabulary from corpus.
    Each word is split into characters, with </w> appended to mark word-end.
    Returns dict {word_tuple: frequency}.
    """
    vocab = defaultdict(int)
    for word in corpus:
        chars = tuple(list(word) + ['</w>'])
        vocab[chars] += 1
    return dict(vocab)


def get_pairs(vocab: dict) -> Counter:
    """
    Count all adjacent symbol pairs in the vocabulary.
    Returns Counter of {pair: count}.
    """
    pairs = Counter()
    for word, freq in vocab.items():
        # TODO: iterate through adjacent symbols and count with their word frequency
        pass
    return pairs


def merge_vocab(pair: tuple, vocab: dict) -> dict:
    """
    Merge all occurrences of pair into a single symbol.
    Returns new vocabulary with the merge applied.
    """
    new_vocab = {}
    bigram = ' '.join(pair)
    merged = ''.join(pair)
    for word, freq in vocab.items():
        # TODO: convert word tuple to string, replace the pair, convert back to tuple
        new_vocab[word] = freq   # placeholder — replace this
    return new_vocab


# Test on a small corpus
corpus = "low low low low lowest lowest newer newer newer newer wider wider new new".split()
vocab = get_vocab(corpus)
print('Initial vocab:')
for w, f in sorted(vocab.items(), key=lambda x: -x[1]):
    print(f'  {" ".join(w):<20} : {f}')

In [ ]:
# Run BPE for num_merges steps
def run_bpe(corpus: list, num_merges: int = 10):
    vocab = get_vocab(corpus)
    merges = []

    for i in range(num_merges):
        pairs = get_pairs(vocab)
        if not pairs:
            break
        best = pairs.most_common(1)[0][0]
        vocab = merge_vocab(best, vocab)
        merges.append(best)
        print(f'Merge {i+1:2d}: {best[0]} + {best[1]} → {"" .join(best)}')

    return vocab, merges


print('BPE merges:')
final_vocab, merges = run_bpe(corpus, num_merges=10)
print('\nFinal vocabulary:')
for w, f in sorted(final_vocab.items(), key=lambda x: -x[1]):
    print(f'  {" ".join(w):<20} : {f}')

## Part C — Using the Library Tokenizer

In [ ]:
# Check if sentencepiece is available
try:
    from src.data.tokenizer import BPETokenizer

    tokenizer = BPETokenizer()
    texts = [
        "once upon a time in a land far far away",
        "the transformer architecture revolutionised natural language processing",
        "attention is all you need to build powerful language models",
    ] * 10   # repeat to give training data

    tokenizer.train(texts, vocab_size=200, model_prefix='workshop_tok')

    test_sentence = "the transformer learns to attend to relevant tokens"
    ids = tokenizer.encode(test_sentence)
    decoded = tokenizer.decode(ids)

    print(f'Input    : {test_sentence}')
    print(f'Token IDs: {ids}')
    print(f'Decoded  : {decoded}')
    print(f'\nVocab size: {tokenizer.vocab_size}')

except ImportError:
    print('sentencepiece not installed. Run: pip install sentencepiece')
    print('Using a character tokenizer for demonstration...')

    # Simple char tokenizer fallback
    text = "hello world"
    chars = sorted(set(text))
    char2id = {c: i for i, c in enumerate(chars)}
    ids = [char2id[c] for c in text]
    print(f'Char tokens: {ids}')

## Part D — Vocabulary Size vs Sequence Length

### Exercise D1 — Analyse the trade-off

In [ ]:
# Demonstrate how vocab size affects sequence length
sample_text = "the quick brown fox jumps over the lazy dog " * 5
words = sample_text.split()

# Character-level: ~4.5 chars per word on average
char_len = len(sample_text.replace(' ', ''))
word_len  = len(words)

# Subword estimate: ~1.3 tokens per word (GPT-style)
subword_estimate = int(word_len * 1.3)

print(f'Text: "{sample_text[:40]}..."')
print(f'\nTokenisation comparison:')
print(f'  Character-level : {char_len:4d} tokens  (vocab ~100)')
print(f'  Subword (BPE)   : {subword_estimate:4d} tokens  (vocab ~30k–100k)')
print(f'  Word-level      : {word_len:4d} tokens  (vocab ~50k–500k, OOV risk)')

# Visualise
methods = ['Character\n(vocab~100)', 'BPE Subword\n(vocab~50k)', 'Word\n(vocab~200k)']
seq_lens = [char_len, subword_estimate, word_len]

plt.figure(figsize=(7, 4))
plt.bar(methods, seq_lens, color=['tomato', 'steelblue', 'seagreen'])
plt.ylabel('Sequence length (tokens)')
plt.title('Tokenisation: Sequence Length vs Method')
for i, v in enumerate(seq_lens):
    plt.text(i, v + 1, str(v), ha='center')
plt.tight_layout(); plt.show()

## Part E — Padding and Attention Masks

When batching sequences of different lengths, we pad shorter sequences and create attention masks to ignore padding.

In [ ]:
# Simulate a batch of variable-length sequences
sequences = [
    [10, 24, 7, 3],
    [5, 18, 2, 41, 9, 15],
    [33, 7],
    [12, 4, 8, 27, 3],
]

PAD = 0
max_len = max(len(s) for s in sequences)

# TODO: pad all sequences to max_len and create attention masks (1=valid, 0=pad)
padded    = []
attn_mask = []
for seq in sequences:
    pad_len = max_len - len(seq)
    # TODO: append padded sequence and mask
    padded.append(seq + [PAD] * pad_len)
    attn_mask.append([1] * len(seq) + [0] * pad_len)

padded    = torch.tensor(padded)
attn_mask = torch.tensor(attn_mask)

print('Padded sequences:')
print(padded)
print('\nAttention mask (1=real, 0=pad):')
print(attn_mask)

## Summary

- **BPE** iteratively merges frequent symbol pairs — a sweet spot between characters and words.
- Modern LLMs use 30k–128k token vocabularies (GPT-4: ~100k, Gemma: 256k).
- **Padding masks** tell attention which tokens are real — padded positions should not contribute to the output.

**Next:** `11_training_loop_starter.ipynb` — training our model end-to-end.